# Installing cuda For run with Gpu only Environment

In [1]:
!pip uninstall -y torch torchvision torchaudio -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 81.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 49.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 96.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 33.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 20.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/905.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

# Installing Dependencies

In [2]:
from ultralytics import YOLO
from roboflow import Roboflow
import os
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA Available: True
GPU Count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


# Importing and Extracting the Dataset

In [4]:
rf = Roboflow(api_key="dgHo0LlAYM7nxKIrclIf")

project = rf.workspace("yolo-trials").project("vehicle-detection-eckrb")
dataset = project.version(4).download("yolov8")

data_yaml = os.path.join(dataset.location, "data.yaml")

print("Dataset path:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Vehicle-detection-4 in yolov8:: 100%|██████████| 12392/12392 [00:01<00:00, 11262.61it/s]

Dataset path: /kaggle/working/Vehicle-detection-4


In [5]:
import yaml

path = "/kaggle/working/Vehicle-detection-4/data.yaml"

with open(path, "r") as f:
    data = yaml.safe_load(f)

print(data)

{'names': ['Ambulance', 'Auto', 'Bicycle', 'Bike', 'Bus', 'Car', 'Motorcycle', 'Truck', 'auto-rickshaw', 'bus', 'car', 'lorry', 'mini truck', 'motorcycle', 'truck'], 'nc': 15, 'roboflow': {'license': 'CC BY 4.0', 'project': 'vehicle-detection-eckrb', 'url': 'https://universe.roboflow.com/yolo-trials/vehicle-detection-eckrb/dataset/4', 'version': 4, 'workspace': 'yolo-trials'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [6]:
clean_names = [
    "ambulance",
    "auto_rickshaw",
    "bicycle",
    "bike",
    "bus",
    "car",
    "truck"
]

In [7]:
import yaml

path = "/kaggle/working/Vehicle-detection-4/data.yaml"

with open(path, "r") as f:
    data = yaml.safe_load(f)

# overwrite classes
data["nc"] = 7
data["names"] = clean_names

with open(path, "w") as f:
    yaml.dump(data, f, default_flow_style=False)

print("data.yaml updated successfully!")
print(data)

data.yaml updated successfully!
{'names': ['ambulance', 'auto_rickshaw', 'bicycle', 'bike', 'bus', 'car', 'truck'], 'nc': 7, 'roboflow': {'license': 'CC BY 4.0', 'project': 'vehicle-detection-eckrb', 'url': 'https://universe.roboflow.com/yolo-trials/vehicle-detection-eckrb/dataset/4', 'version': 4, 'workspace': 'yolo-trials'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


# Loading the Model 

In [8]:
model = YOLO("yolov8m.pt")

# Model Training

In [9]:
model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=32,
    device=[0, 1],
    workers=8,
    amp=True,
    name="vehicle_detector"
)

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Vehicle-detection-4/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicl

# Metrics

In [10]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 762.4±317.8 MB/s, size: 18.9 KB)
val: Scanning /kaggle/working/Vehicle-detection-4/valid/labels.cache... 1233 images, 0 backgrounds, 313 corrupt: 100% ━━━━━━━━━━━━ 1233/1233 287.3Mit/s 0.0s
val: /kaggle/working/Vehicle-detection-4/valid/images/104Truck_jpg.rf.f5bc19f5ccafc58ed69f481abda35d6c.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 7. Possible class labels are 0-6
val: /kaggle/working/Vehicle-detection-4/valid/images/105Truck_jpg.rf.81f9d9c223fe98535550364a55e3aff2.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 7. Possible class labels are 0-6
val: /kaggle/working/Vehicle-detection-4/valid/images/115Truck_jpg.rf.e6c60a2a7aeac8813c9485074970957b.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset cl

In [11]:
best_model_path = "runs/detect/vehicle_detector/weights/best.pt"
print("Best model saved at:", best_model_path)

Best model saved at: runs/detect/vehicle_detector/weights/best.pt


# Testing on an unknown Sample image

In [12]:
model = YOLO("/kaggle/working/runs/detect/vehicle_detector/weights/best.pt")

results = model(
    "https://ultralytics.com/images/bus.jpg",
    save=True,
    conf=0.25
)

print("Image prediction done!")


image 1/1 /kaggle/working/bus.jpg: 640x480 1 ambulance, 1 bus, 68.8ms
Speed: 3.9ms preprocess, 68.8ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /kaggle/working/runs/detect/predict
Image prediction done!


# Metrics of the trained model

In [13]:
from ultralytics import YOLO
model = YOLO("/kaggle/working/runs/detect/vehicle_detector/weights/best.pt")
metrics = model.val()

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 888.5±331.9 MB/s, size: 22.2 KB)
val: Scanning /kaggle/working/Vehicle-detection-4/valid/labels.cache... 1233 images, 0 backgrounds, 313 corrupt: 100% ━━━━━━━━━━━━ 1233/1233 431.0Mit/s 0.0s
val: /kaggle/working/Vehicle-detection-4/valid/images/104Truck_jpg.rf.f5bc19f5ccafc58ed69f481abda35d6c.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 7. Possible class labels are 0-6
val: /kaggle/working/Vehicle-detection-4/valid/images/105Truck_jpg.rf.81f9d9c223fe98535550364a55e3aff2.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 7. Possible class labels are 0-6
val: /kaggle/working/Vehicle-detection-4/valid/images/115Truck_jpg.rf.e6c60a2a7aeac8813c9485074970957b.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset cl

# Checking scores and Accuracy

In [14]:
precision = metrics.box.mp      
recall = metrics.box.mr 
map50 = metrics.box.map50
map95 = metrics.box.map

In [15]:
f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)

In [16]:
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)
print("mAP@50:", map50)
print("mAP@50-95:", map95)

Precision: 0.8420817527014758
Recall: 0.7086670600006025
F1 Score: 0.7696349147736505
mAP@50: 0.7996347294367881
mAP@50-95: 0.7981591921639956


# No. of Classes in total

In [17]:
!cat /kaggle/working/Vehicle-detection-4/data.yaml

names:
- ambulance
- auto_rickshaw
- bicycle
- bike
- bus
- car
- truck
nc: 7
roboflow:
  license: CC BY 4.0
  project: vehicle-detection-eckrb
  url: https://universe.roboflow.com/yolo-trials/vehicle-detection-eckrb/dataset/4
  version: 4
  workspace: yolo-trials
test: ../test/images
train: ../train/images
val: ../valid/images


In [18]:
import yaml

with open("/kaggle/working/Vehicle-detection-4/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print("Number of classes:", data["nc"])
print("Classes:", data["names"])

Number of classes: 7
Classes: ['ambulance', 'auto_rickshaw', 'bicycle', 'bike', 'bus', 'car', 'truck']


In [19]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/vehicle_detector/weights/best.pt")

print(model.names)   # class dictionary
print("Total classes:", len(model.names))

{0: 'ambulance', 1: 'auto_rickshaw', 2: 'bicycle', 3: 'bike', 4: 'bus', 5: 'car', 6: 'truck'}
Total classes: 7
